## Harmony Py Library
### Paging Through Job Results

`result_urls()` accepts optional `offset` and `limit` arguments so you can page through a
job's output links instead of pulling the complete list at once. It also accepts an
`allow_incomplete` flag to return whatever links are available so far, without waiting for
the job to reach a terminal state.

This example uses `max_results=10` on the request so the job only produces a handful of
granules, keeping the paging examples below small and easy to read.

In [ ]:
import sys
import helper
helper.install_project_and_dependencies('..')

from harmony import BBox, Client, Collection, Request, Environment

#### Submit a request for processing and return the job_id

In [ ]:
harmony_client = Client(env=Environment.UAT)  # assumes .netrc usage

collection = Collection(id='C1234088182-EEDTEST')
request = Request(
    collection=collection,
    spatial=BBox(-165, 52, -140, 77),
    max_results=10,  # keep the job small so paging is easy to see
)

job_id = harmony_client.submit(request)
job_id

#### Default behavior: no offset/limit means every data URL is returned. Blocking - waits for the job to reach a terminal state.

In [ ]:
all_urls = list(harmony_client.result_urls(job_id))
len(all_urls), all_urls

#### 'limit' caps how many URLs are returned

In [ ]:
first_page = list(harmony_client.result_urls(job_id, limit=3))
first_page

#### 'page' skips past URLs already seen, so the next page picks up where the last one left off.

In [ ]:
second_page = list(harmony_client.result_urls(job_id, page=2, limit=3))
second_page

#### A simple paging loop: keep requesting pages until an empty page comes back.

In [ ]:
page_size = 3
page = 1
while True:
    results = list(harmony_client.result_urls(job_id, page=page, limit=page_size))
    if not results:
        break
    print(f'page={page}: {results}')
    page += 1

#### Asking for a page/limit beyond the available results returns an empty list rather than raising an error.

In [ ]:
list(harmony_client.result_urls(job_id, page=1000, limit=10))

#### 'allow_incomplete=True' returns whatever data URLs are currently available without waiting for the job to finish processing. This is useful for a still-running job when you just want to start downloading what's ready so far. Combine with offset/limit to page through the partial results too.

In [ ]:
partial_job_id = harmony_client.submit(request)
partial_job_id

#### Run this cell while the request is running to see the partial results returned without blocking

In [ ]:
partial_urls = list(harmony_client.result_urls(partial_job_id, allow_incomplete=True))
len(partial_urls), partial_urls